# 1.3 Exploración de P2 para la etapa 2

P2: ¿una estación estará en riesgo aproximado de quedarse sin bicicletas o sin anclajes en la siguiente hora? En esta etapa se comparan el baseline y tres candidatos del plan de `config.py` antes de empaquetarlos para MLflow. Todo se decide con **validación** (junio y julio de 2026); prueba no se usa aquí, se mide una sola vez en la corrida oficial.

In [1]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, precision_recall_curve

from mibici_equipo39 import config as C
from mibici_equipo39.features import load_p2_targets, load_tables, proxy_labels
from mibici_equipo39.modeling.p2 import COLUMNS, choose_threshold, classification_metrics, make_baseline, make_candidate, threshold_search

metadata, X, _ = load_tables()
df = pd.concat([metadata, X, load_p2_targets()], axis=1)
df = df.loc[df["capacity"].notna()].reset_index(drop=True)
for target in ["y_bikes", "y_docks"]:
    df[target] = df[target].astype(int)
train = df["split"].eq("train").to_numpy()
val = df["split"].eq("validation").to_numpy()
print(df["station_id"].nunique(), "estaciones;", train.sum(), "filas train;", val.sum(), "filas validación")

469 estaciones; 2735208 filas train; 686616 filas validación


## Partición y prevalencia

In [2]:
df.groupby("split").agg(desde=("prediction_time", "min"), hasta=("prediction_time", "max"),
                        riesgo_bicis=("y_bikes", "mean"), riesgo_anclajes=("y_docks", "mean")).round(4)

,desde,hasta,riesgo_bicis,riesgo_anclajes
split,,,,
test,2026-08-01,2026-09-30 23:00:00,0.0044,0.0057
train,2025-10-01,2026-05-31 23:00:00,0.0040,0.0052
validation,2026-06-01,2026-07-31 23:00:00,0.0032,0.0043


> **Partición cronológica.** Train va de octubre a mayo, validación de junio a julio y prueba de agosto a septiembre. Ningún modelo ve fechas posteriores a las que usa para entrenar, así que la validación simula elegir un modelo para meses que todavía no ocurren. Menos de 1% de las horas son positivas, por eso no sirve la exactitud: se miden recall, precisión, F1 y AP.

## Variantes del plan

In [3]:
variants = {"baseline": None, **{v["name"]: v["params"]["C"] for v in C.STAGE2_SEARCH_PLAN["p2"]}}
pd.DataFrame([{"variante": k, "modelo": "DummyClassifier(prior)" if c is None else f"LogisticRegression(C={c}, balanced, liblinear)"}
              for k, c in variants.items()])

,variante,modelo
0,baseline,DummyClassifier(prior)
1,p2_lr_0.1,"LogisticRegression(C=0.1, balanced, liblinear)"
2,p2_lr_1,"LogisticRegression(C=1.0, balanced, liblinear)"
3,p2_lr_10,"LogisticRegression(C=10.0, balanced, liblinear)"


> **Por qué variar C.** C controla la regularización: con C pequeño los coeficientes se encogen y el modelo es más simple; con C grande se ajusta más a train. Con 469 estaciones en one-hot hay muchos coeficientes y pocos positivos por estación, así que probar tres órdenes de magnitud dice si el modelo está sobreajustando o si le faltan señales. `class_weight="balanced"` se mantiene porque compensa la clase rara y deja los puntajes en el rango de la rejilla 0.05 a 0.95.

## Ajuste en train y umbral en validación

In [4]:
results, scores, costs = {}, {}, {}
for name, c in variants.items():
    start = time.perf_counter()
    for risk, target in [("bikes", "y_bikes"), ("docks", "y_docks")]:
        model = make_baseline() if c is None else make_candidate().set_params(logisticregression__C=c)
        model.fit(df.loc[train, COLUMNS], df.loc[train, target])
        score = model.predict_proba(df.loc[val, COLUMNS])[:, 1]
        y = df.loc[val, target].to_numpy()
        threshold, rule = choose_threshold(y, score)
        values, _ = classification_metrics(y, score, threshold)
        scores[(name, risk)] = (y, score, threshold)
        results[(name, risk)] = {"umbral": threshold, "regla": rule, "recall": values["recall"], "precision": values["precision"],
                                 "f1": values["f1"], "ap": values.get("ap"), "ap_vs_prevalencia": values.get("ap", 0) / y.mean(),
                                 "alertas_por_1000": 1000 * (score >= threshold).mean()}
    costs[name] = time.perf_counter() - start
table = pd.DataFrame(results).T
table.round(4)

umbral         regla    recall precision        f1        ap  \
baseline  bikes   0.95        max_f1       0.0       0.0       0.0  0.003223   
          docks   0.95        max_f1       0.0       0.0       0.0  0.004261   
p2_lr_0.1 bikes   0.65  recall_floor  0.808857  0.017187  0.033658  0.030979   
          docks    0.7  recall_floor  0.807587  0.026364  0.051062  0.051567   
p2_lr_1   bikes   0.65  recall_floor  0.813376  0.017116  0.033527  0.030742   
          docks    0.7  recall_floor  0.811347  0.026097  0.050568  0.051352   
p2_lr_10  bikes   0.65  recall_floor  0.813376  0.017094  0.033484   0.03072   
          docks    0.7  recall_floor  0.811347  0.026053  0.050485  0.051317   

                ap_vs_prevalencia alertas_por_1000  
baseline  bikes               1.0              0.0  
          docks               1.0              0.0  
p2_lr_0.1 bikes          9.611646       151.687406  
          docks         12.100824       130.537302  
p2_lr_1   bikes          9.538215       153.159845  
          docks         12.050273       132.485989  
p2_lr_10  bikes           9.53118       153.363743  
          docks         12.042124       132.710278

In [5]:
summary = table.groupby(level=0)["f1"].mean().rename("val_macro_f1").to_frame()
summary["ajuste_s"] = pd.Series(costs)
summary.round(4).sort_values("val_macro_f1", ascending=False)

,val_macro_f1,ajuste_s
p2_lr_0.1,0.04236,11.2467
p2_lr_1,0.042048,14.4981
p2_lr_10,0.041985,17.2614
baseline,0.0,0.6208


> **Lectura.** El dummy nunca alcanza la rejilla, así que cae al respaldo de mayor F1 y su F1 es 0; su AP es igual a la prevalencia. Las tres logísticas detectan alrededor de 81% de los riesgos y ordenan unas 10 veces (bicicletas) y 12 veces (anclajes) mejor que el azar. Entre ellas la diferencia de macro F1 es de unas 4 diezmilésimas: **C casi no cambia el resultado**, lo que indica que el límite está en las variables (calendario y estación), no en la regularización.

## Umbrales del mejor candidato

In [6]:
best = summary["val_macro_f1"].idxmax()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, risk in zip(axes, ["bikes", "docks"]):
    y, score, threshold = scores[(best, risk)]
    grid = threshold_search(y, score)
    ax.plot(grid["threshold"], grid["recall"], marker="o", label="recall")
    ax.plot(grid["threshold"], grid["precision"], marker="o", label="precisión")
    ax.axhline(C.P2_MIN_RECALL, color="grey", linestyle=":", label="recall mínimo")
    ax.axvline(threshold, color="black", linestyle="--", label=f"umbral {threshold}")
    ax.set(title=f"{best}: {risk} en validación", xlabel="umbral", ylabel="valor")
    ax.legend(fontsize=8)
plt.tight_layout()

> **Por qué esos umbrales.** La rejilla va de 0.05 a 0.95. Se queda el umbral de mayor precisión entre los que detectan al menos 80% de los riesgos; en empate, el más alto. Subir el umbral hace que caiga el recall por debajo de 0.80, así que el umbral elegido es el último que cumple el mínimo.

## Curvas precisión-recall

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, risk in zip(axes, ["bikes", "docks"]):
    for name in variants:
        y, score, _ = scores[(name, risk)]
        precision, recall, _ = precision_recall_curve(y, score)
        ax.plot(recall, precision, label=name)
    ax.axvline(C.P2_MIN_RECALL, color="grey", linestyle="--")
    ax.set(title=f"Precisión-recall {risk} (validación)", xlabel="recall", ylabel="precisión", ylim=(0, 0.25))
    ax.legend(fontsize=8)
plt.tight_layout()

## Matrices de confusión del mejor candidato

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, risk in zip(axes, ["bikes", "docks"]):
    y, score, threshold = scores[(best, risk)]
    ConfusionMatrixDisplay(confusion_matrix(y, score >= threshold), display_labels=["sin riesgo", "riesgo"]).plot(ax=ax, colorbar=False)
    ax.set_title(f"{best}: {risk}, validación, umbral {threshold}")
plt.tight_layout()

> **Costo operativo.** Para detectar 4 de cada 5 riesgos el modelo marca muchas horas sin riesgo: los falsos positivos superan por mucho a los verdaderos. Por eso el resultado sirve para **ordenar** visitas de rebalanceo, no para mandar un camión por cada alerta.

## Estabilidad por mes y costo

In [9]:
val_month = df.loc[val, "prediction_time"].dt.month.to_numpy()
stability = {}
for name in variants:
    for month in (6, 7):
        f1 = [classification_metrics(scores[(name, r)][0][val_month == month], scores[(name, r)][1][val_month == month],
                                     scores[(name, r)][2])[0]["f1"] for r in ("bikes", "docks")]
        stability.setdefault(name, {})[f"macro_f1_{'junio' if month == 6 else 'julio'}"] = np.mean(f1)
pd.concat([pd.DataFrame(stability).T, summary], axis=1).round(4)

,macro_f1_junio,macro_f1_julio,val_macro_f1,ajuste_s
baseline,0.0000,0.0000,0.0,0.6208
p2_lr_0.1,0.0418,0.0429,0.04236,11.2467
p2_lr_1,0.0414,0.0426,0.042048,14.4981
p2_lr_10,0.0414,0.0425,0.041985,17.2614


> **Estabilidad.** El macro F1 casi no cambia entre junio y julio en ninguna logística, así que el resultado no depende de un mes atípico. El tiempo de ajuste crece con C porque `liblinear` necesita más iteraciones cuando regulariza menos.

## Sensibilidad al supuesto de inventario

In [10]:
sens = {}
for occupancy in (0.25, 0.50, 0.75):
    bikes, docks = proxy_labels(df["flow_min"], df["flow_max"], df["capacity"], occupancy=occupancy)
    sens[f"{int(occupancy * 100)}%"] = {"riesgo_bicis": bikes.mean(), "riesgo_anclajes": docks.mean()}
pd.DataFrame(sens).T.round(4)

,riesgo_bicis,riesgo_anclajes
25%,0.0554,0.0006
50%,0.0040,0.0051
75%,0.0004,0.0617


> **La etiqueta es una aproximación.** Con 25% de ocupación inicial el riesgo de bicicletas se multiplica y con 75% el de anclajes; la etiqueta principal se queda en 50% y no se ajusta a conveniencia. MiBici no publica inventario, así que los puntajes ordenan presión de flujo, no prueban que una estación se vació.

## Conclusión preliminar

| | |
|---|---|
| Mejor en validación | La logística de menor C, con la mayor `val_macro_f1` |
| Contra el baseline | Pasa de F1 0 a unos 4 puntos porcentuales de macro F1, con 81% de recall en ambos riesgos |
| Entre candidatos | Diferencias menores a una milésima; C no es la palanca |
| Costo | El de menor C también es el más rápido de ajustar |
| Límite | Precisión de 1.7% (bicicletas) y 2.6% (anclajes); alerta entre 13 y 15% de las estación-horas |

La selección oficial la hace el controlador con la corrida de Marco, usando esta misma regla. El siguiente paso es `2.3-dm-p2-refactorizacion-a6`: el `ADAPTER` de P2 y el modelo `pyfunc` que junta los dos riesgos y sus umbrales.